```
2. HTTP API
Реализуйте два endpoint'а:
GET /health - возвращает {"status": "ok"}
POST /recommend - принимает JSON в формате:

{
  "user_id": "string",
  "watched_videos": ["video_id1", "video_id2"],
  "liked_categories": ["adventure", "education"]
}
Возвращает JSON:
{
  "user_id": "string",
  "recommendations": ["video_id3", "video_id4", "video_id5"],
  "algorithm_version": "1.0"
}


3. ML алгоритм 
Реализуйте простую рекомендательную систему:
Используйте предобученные эмбеддинги (предоставлены в задании)

Алгоритм должен рекомендовать видео на основе:
Категорий, которые нравятся пользователю
Видео, которые пользователь уже смотрел (исключить их из рекомендаций)
Достаточно использовать косинусное сходство или простой rule-based подход
Для данных используйте файл videos_data.csv (формат см. ниже)

```

In [1]:
ls

app/                Dockerfile     README.md
docker-compose.yml  ml_algo.ipynb  requirements.txt


In [2]:
import pandas as pd

df = pd.read_csv('videos_data.csv',sep=';')
df

,video_id,title,category,embedding
0,v1,Mountain Adventure,adventure,"""[0.1, 0.2, 0.3]"""
1,v2,Space Exploration,science,"""[0.4, 0.5, 0.6]"""
2,v3,Ancient History,education,"""[0.7, 0.8, 0.9]"""
3,v4,Ocean Wonders,nature,"""[0.2, 0.3, 0.4]"""
4,v5,Robot Friends,science,"""[0.5, 0.6, 0.7]"""
5,v6,Ancient Egypt,education,"""[0.8, 0.9, 1.0]"""


# Алгоритм рекомендации
## 1. Чтение данных

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json

top_n=3

user_data = {
    "watched_videos": ["v1", "v2"],
    "liked_categories": ["adventure", "education"]
}


# Читаем эмбеддинги
df = pd.read_csv('videos_data.csv',sep=';')

# Преобразуем эмбеддинги из строки в массив numpy
df['embedding'] = df['embedding'].apply(lambda x: np.array(json.loads(json.loads(x))))

# Исключаем просмотренные видео
available_videos = df[~df['video_id'].isin(user_data['watched_videos'])].copy()

# Вычисляем эмбеддинг пользователя
user_embedding = np.zeros(3)   

df

,video_id,title,category,embedding
0,v1,Mountain Adventure,adventure,"[0.1, 0.2, 0.3]"
1,v2,Space Exploration,science,"[0.4, 0.5, 0.6]"
2,v3,Ancient History,education,"[0.7, 0.8, 0.9]"
3,v4,Ocean Wonders,nature,"[0.2, 0.3, 0.4]"
4,v5,Robot Friends,science,"[0.5, 0.6, 0.7]"
5,v6,Ancient Egypt,education,"[0.8, 0.9, 1.0]"


In [4]:
available_videos

,video_id,title,category,embedding
2,v3,Ancient History,education,"[0.7, 0.8, 0.9]"
3,v4,Ocean Wonders,nature,"[0.2, 0.3, 0.4]"
4,v5,Robot Friends,science,"[0.5, 0.6, 0.7]"
5,v6,Ancient Egypt,education,"[0.8, 0.9, 1.0]"


In [5]:
user_embedding

array([0., 0., 0.])

## 2. Расчет user_embedding

In [6]:

# Учитываем просмотренные видео (среднее их эмбеддингов)
watched_mask = df['video_id'].isin(user_data['watched_videos'])
if watched_mask.any():
    watched_embeddings = np.vstack(df[watched_mask]['embedding'].values)     
    user_embedding += watched_embeddings.mean(axis=0)


# Учитываем любимые категории (среднее эмбеддингов видео этих категорий)
for category in user_data['liked_categories']:
    category_mask = df['category'] == category
    
    if category_mask.any():
        category_embeddings = np.vstack(df[category_mask]['embedding'].values)
        user_embedding += category_embeddings.mean(axis=0)

user_embedding = user_embedding.reshape(1, -1)
user_embedding

array([[1.1, 1.4, 1.7]])

## 3. Расчет рекомендации (cos sim)

In [7]:


# Вычисляем косинусное сходство
available_embeddings = np.vstack(available_videos['embedding'].values)
similarities = cosine_similarity(user_embedding, available_embeddings)[0]

available_videos['similarity'] = similarities

# Сортируем по сходству и выбираем топ-N
recommendations = available_videos.sort_values('similarity', ascending=False).head(top_n)

recommendations

,video_id,title,category,embedding,similarity
4,v5,Robot Friends,science,"[0.5, 0.6, 0.7]",0.999280
2,v3,Ancient History,education,"[0.7, 0.8, 0.9]",0.997445
5,v6,Ancient Egypt,education,"[0.8, 0.9, 1.0]",0.996579


In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json

top_n=3

user_data = {
    "watched_videos": ["v1", "v2"],
    "liked_categories": ["adventure", "education"]
}


def algo_recommendations(user_data: dict, top_n: int = 3):
        
    
    # Читаем эмбеддинги
    df = pd.read_csv('videos_data.csv',sep=';')
    
    # Преобразуем эмбеддинги из строки в массив numpy
    df['embedding'] = df['embedding'].apply(lambda x: np.array(json.loads(json.loads(x))))
    
    # Исключаем просмотренные видео
    available_videos = df[~df['video_id'].isin(user_data['watched_videos'])].copy()
    
    # Вычисляем эмбеддинг пользователя
    user_embedding = np.zeros(3)   
    
    
    # Учитываем просмотренные видео (среднее их эмбеддингов)
    watched_mask = df['video_id'].isin(user_data['watched_videos'])
    if watched_mask.any():
        watched_embeddings = np.vstack(df[watched_mask]['embedding'].values)     
        user_embedding += watched_embeddings.mean(axis=0)
    
    
    # Учитываем любимые категории (среднее эмбеддингов видео этих категорий)
    for category in user_data['liked_categories']:
        category_mask = df['category'] == category
        
        if category_mask.any():
            category_embeddings = np.vstack(df[category_mask]['embedding'].values)
            user_embedding += category_embeddings.mean(axis=0)
    
    user_embedding = user_embedding.reshape(1, -1)

    # Вычисляем косинусное сходство
    available_embeddings = np.vstack(available_videos['embedding'].values)
    similarities = cosine_similarity(user_embedding, available_embeddings)[0]
    
    available_videos['similarity'] = similarities
    
    # Сортируем по сходству и выбираем топ-N
    recommendations = available_videos.sort_values('similarity', ascending=False).head(top_n)
    
    return recommendations["video_id"].to_list()


algo_recommendations(user_data)

['v5', 'v3', 'v6']

## Функция для сервиса

In [9]:
import csv
import json
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity


@dataclass
class Video:
    video_id: str
    title: str
    category: str
    embedding: np.ndarray
    similarity: float = 0.0


def parse_embedding(embedding_str: str) -> np.ndarray:
    """Parse embedding string to numpy array."""
    try:
        # Try parsing as double-encoded JSON
        embedding_str = json.loads(embedding_str)
        embedding_list = json.loads(embedding_str)
    except json.JSONDecodeError:
        # If that fails, try parsing directly
        embedding_list = json.loads(embedding_str)
    return np.array(embedding_list)


def read_videos_from_csv(filepath: str) -> List[Video]:
    """Read video data from CSV file."""
    videos = []
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            embedding = parse_embedding(row['embedding'])
            video = Video(
                video_id=row['video_id'],
                title=row['title'],
                category=row['category'],
                embedding=embedding
            )
            videos.append(video)
    return videos


def algo_recommendations(user_data: dict, top_n: int = 3) -> List[str]:
    """Generate video recommendations based on user data."""
    videos = read_videos_from_csv('videos_data.csv')
    
    watched_videos = set(user_data.get('watched_videos', []))
    liked_categories = set(user_data.get('liked_categories', []))
    
    available_videos = []
    watched_embeddings = []
    category_embeddings = {category: [] for category in liked_categories}
    
    for video in videos:
        if video.category in liked_categories:
            category_embeddings[video.category].append(video.embedding)
        
        if video.video_id in watched_videos:
            watched_embeddings.append(video.embedding)
        else:
            available_videos.append(video)
    
    user_embedding = np.zeros(3)
    
    if watched_embeddings:
        user_embedding += np.vstack(watched_embeddings).mean(axis=0)
    
    for embeddings in category_embeddings.values():
        if embeddings:
            user_embedding += np.vstack(embeddings).mean(axis=0)
    
    user_embedding = user_embedding.reshape(1, -1)
    
    if not available_videos:
        return []
    
    available_embeddings = np.vstack([v.embedding for v in available_videos])
    similarities = cosine_similarity(user_embedding, available_embeddings)[0]
    
    for video, similarity in zip(available_videos, similarities):
        video.similarity = similarity
    
    available_videos.sort(key=lambda x: x.similarity, reverse=True)
    
    return [video.video_id for video in available_videos[:top_n]]

algo_recommendations(user_data)

['v5', 'v3', 'v6']